# 05. Implementation-Fidelity Validation

[![License](https://img.shields.io/badge/license-MIT-green)](../LICENSE) [![Python](https://img.shields.io/badge/python-3.10%2B-blue)](../requirements.txt)

**Source script:** `src/fidelity.py` &nbsp;|&nbsp; **Notebook 5 of 10**

Cross-validates the six metaphor-based implementations against the independent `mealpy` library.

Part of *Budget-Controlled Reproduction Study of Nature-Inspired Metaheuristics* — a reproduction study comparing six metaphor-based metaheuristics (GWO, WOA, SCA, SSA, HHO, AOA) against five established baselines (DE, PSO, L-SHADE, CMA-ES, random search) on constrained engineering design problems, under matched evaluation budgets and tuning effort.

See the [repository README](../README.md) for installation and full reproduction instructions, and [notebooks/README.md](README.md) for the notebook index and suggested run order.

---


# Implementation-Fidelity Validation Against mealpy

![Python](https://img.shields.io/badge/python-3.10%2B-blue) ![Status](https://img.shields.io/badge/status-research--reproduction-lightgrey) ![License](https://img.shields.io/badge/license-MIT-green)

Cross-checks the six metaphor-based algorithm implementations against an independent third-party library (`mealpy`) under an identical evaluation budget and constraint handler, using a two-sided Wilcoxon rank-sum test per algorithm-problem pair.


## Imports

External libraries and project modules used by this notebook.

In [ ]:
import json
import os
import warnings

import numpy as np
from scipy import stats

In [ ]:
warnings.filterwarnings("ignore")

from problems import PROBLEMS, PROBLEM_MAP          # noqa: E402
from algorithms import run_one                      # noqa: E402
from mealpy import FloatVar, GWO, WOA, SCA, SSA, HHO, AOA   # noqa: E402

BUDGET = 15000
N_RUNS = 9
POP = 30
PENALTY_C = 1e5
OUT = os.path.join(os.path.dirname(__file__), "..", "results")

REF = {"GWO": GWO.OriginalGWO, "WOA": WOA.OriginalWOA, "SCA": SCA.OriginalSCA,
       "SSA": SSA.OriginalSSA, "HHO": HHO.OriginalHHO, "AOA": AOA.OriginalAOA}

### `penalised`

Builds a scalar penalised objective for the reference `mealpy` implementation.


In [ ]:
def penalised(problem):
    def obj(x):
        X = problem.repair(np.array(x, dtype=float).reshape(1, -1))
        f, G = problem.evaluate(X)
        return float(f[0] + PENALTY_C * np.maximum(G[0], 0.0).sum())
    return obj

### `run_reference`

Runs the independent `mealpy` implementation of an algorithm under the matched evaluation budget.


In [ ]:
def run_reference(alg, problem, seed):
    np.random.seed(seed)
    prob = {"obj_func": penalised(problem),
            "bounds": FloatVar(lb=list(problem.lb), ub=list(problem.ub)),
            "minmax": "min", "log_to": None}
    model = REF[alg](epoch=100000, pop_size=POP)
    g = model.solve(prob, termination={"max_fe": BUDGET}, seed=seed)
    x = problem.repair(np.array(g.solution, dtype=float).reshape(1, -1))
    f, G = problem.evaluate(x)
    viol = float(np.maximum(G[0], 0.0).sum())
    return (float(f[0]) if viol <= 1e-8 else np.nan), model.nfe_counter

### `main`

Loops over problems and algorithms, compares this study's implementation against the reference, and writes `fidelity.json`.


In [ ]:
def main():
    import sys
    subset = sys.argv[1].split(",") if len(sys.argv) > 1 else list(REF)
    fn = f"{OUT}/fidelity.json"
    rows = json.load(open(fn)) if os.path.exists(fn) else []
    done = {(r["problem"], r["algorithm"]) for r in rows}
    for p in PROBLEMS:
        for alg in subset:
            if (p.name, alg) in done:
                continue
            mine, ref, nfes = [], [], []
            for r in range(N_RUNS):
                res = run_one(alg, p, BUDGET, 90000 + r, scheme="static")
                mine.append(res["best_f"] if res["feasible"] else np.nan)
                fv, nfe = run_reference(alg, p, 90000 + r)
                ref.append(fv); nfes.append(nfe)
            a = np.array(mine, dtype=float); b = np.array(ref, dtype=float)
            ok = np.isfinite(a) & np.isfinite(b)
            if ok.sum() >= 5:
                _, pv = stats.ranksums(a[ok], b[ok])
            else:
                pv = np.nan
            ma, mb = np.nanmedian(a), np.nanmedian(b)
            rel = abs(ma - mb) / max(abs(mb), 1e-12)
            rows.append(dict(problem=p.name, algorithm=alg,
                             median_this_study=float(ma), median_reference=float(mb),
                             rel_diff=float(rel), p_ranksum=float(pv),
                             mean_nfe_reference=float(np.mean(nfes))))
            json.dump(rows, open(fn, "w"), indent=1)
            print(f"{p.name:<17}{alg:<5} this={ma:<13.6g} ref={mb:<13.6g} "
                  f"reldiff={rel:<9.3g} p={pv:.3f}", flush=True)

    agree = sum(1 for r in rows if not (r["p_ranksum"] < 0.05))
    print(f"\nno significant difference on {agree}/{len(rows)} algorithm-problem pairs")

In [ ]:
if __name__ == "__main__":
    main()

---
## Key outcomes

- Across 48 algorithm-problem pairs (6 metaphor-based algorithms x 8 problems), the two independent
  implementations (this study vs. `mealpy`) show **no statistically significant difference on 19/48
  pairs** (Wilcoxon rank-sum, alpha = 0.05); the remainder differ significantly, most often for
  **WOA** (median relative difference 0.35) and **AOA** (0.11), and least for **SSA** (0.006).
- Overall median relative difference between the two implementations is 0.062, i.e. results are of the
  same order of magnitude but not numerically identical - expected given independent re-implementations
  of metaphor-described pseudocode.
- Confirms the accounting caveat noted in the script: `mealpy`'s SSA and HHO consume ~1.9x the nominal
  epoch x pop_size evaluations internally, so termination must be driven by `max_fe`, not epoch count.

*Part of the budget-controlled reproduction study of nature-inspired metaheuristics.*


---


**Citation:** [CITATION.cff](../CITATION.cff) &nbsp;|&nbsp; **License:** [MIT](../LICENSE)

[![License](https://img.shields.io/badge/license-MIT-green)](../LICENSE)
